In [ ]:
from google.colab import drive
drive.mount('/content/drive')

caminho_tsv = '/content/drive/MyDrive/Colab_Notebooks/dataset/val_labels/participants.tsv'
caminho_imagens = '/content/drive/MyDrive/Colab_Notebooks/dataset/val_vbm'

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

#arquiteturas
class CNN3D_OpenBHB(nn.Module):
    def __init__(self, n_classes=1):
        super(CNN3D_OpenBHB, self).__init__()
        self.conv1 = nn.Conv3d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv3d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv3d(128, 256, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv3d(256, 256, kernel_size=3, stride=1, padding=1)
        self.conv6 = nn.Conv3d(256, 64, kernel_size=1, stride=1)

        self.batchnorm1 = nn.BatchNorm3d(32)
        self.batchnorm2 = nn.BatchNorm3d(64)
        self.batchnorm3 = nn.BatchNorm3d(128)
        self.batchnorm4 = nn.BatchNorm3d(256)
        self.batchnorm5 = nn.BatchNorm3d(256)
        self.batchnorm6 = nn.BatchNorm3d(64)

        self.maxpool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.avgpool = nn.AvgPool3d(kernel_size=(3, 4, 3), stride=1)
        self.dropout = nn.Dropout3d(p=0.5)
        self.relu = nn.ReLU()
        self.classifier = nn.Conv3d(64, n_classes, kernel_size=1, stride=1)

    def forward(self, x):
        x = self.relu(self.maxpool(self.batchnorm1(self.conv1(x))))
        x = self.relu(self.maxpool(self.batchnorm2(self.conv2(x))))
        x = self.relu(self.maxpool(self.batchnorm3(self.conv3(x))))
        x = self.relu(self.maxpool(self.batchnorm4(self.conv4(x))))
        x = self.relu(self.maxpool(self.batchnorm5(self.conv5(x))))

        x = self.relu(self.batchnorm6(self.conv6(x)))
        x = self.avgpool(x)
        x = self.dropout(x)
        x = self.classifier(x)
        x = x.view(x.shape[0], -1)
        return x

class OpenBHBDataset(Dataset):
    def __init__(self, tsv_file, img_dir):
        df_completo = pd.read_csv(tsv_file, sep='\t')
        self.img_dir = img_dir

        print("Filtrando pacientes com o nome real dos arquivos...")

        pacientes_validos = []
        for index, row in df_completo.iterrows():
            patient_id = row['participant_id']
            img_name = f"sub-{patient_id}_preproc-cat12vbm_desc-gm_T1w.npy"
            img_path = os.path.join(self.img_dir, img_name)

            if os.path.exists(img_path):
                pacientes_validos.append(row)

        self.labels_df = pd.DataFrame(pacientes_validos)
        print(f"Filtro concluído! Encontradas {len(self.labels_df)} imagens na pasta de um total de {len(df_completo)}.")

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        patient_id = self.labels_df.iloc[idx]['participant_id']
        age = self.labels_df.iloc[idx]['age']
        y_label = torch.tensor([age], dtype=torch.float32)

        img_name = f"sub-{patient_id}_preproc-cat12vbm_desc-gm_T1w.npy"
        img_path = os.path.join(self.img_dir, img_name)

        image_np = np.load(img_path)
        image_np = np.squeeze(image_np)

        image_tensor = torch.from_numpy(image_np).float()
        image_tensor = image_tensor.unsqueeze(0)

        return image_tensor, y_label

#treinamento e validação
def treinar_modelo():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Treinando na unidade: {device}")

    dataset = OpenBHBDataset(tsv_file=caminho_tsv, img_dir=caminho_imagens)

    #separação automatica em 80% treinamento e 20% validação
    tamanho_total = len(dataset)
    tamanho_treino = int(0.8 * tamanho_total)
    tamanho_val = tamanho_total - tamanho_treino

    dataset_treino, dataset_val = random_split(dataset, [tamanho_treino, tamanho_val])

    print(f"Imagens para Treino: {tamanho_treino} | Imagens para Validação: {tamanho_val}")

    dataloader_treino = DataLoader(dataset_treino, batch_size=4, shuffle=True)
    dataloader_val = DataLoader(dataset_val, batch_size=4, shuffle=False)

    modelo = CNN3D_OpenBHB().to(device)
    criterio = nn.L1Loss()
    otimizador = optim.Adam(modelo.parameters(), lr=0.0001)

    num_epocas = 25
    melhor_erro_val = float('inf') #rastrear o melhor modelo

    print("Iniciando o Treinamento...")
    for epoca in range(num_epocas):

        # ---------------- FASE DE TREINAMENTO ----------------
        modelo.train()
        erro_treino_total = 0.0

        loop_treino = tqdm(dataloader_treino, total=len(dataloader_treino), leave=False)

        for imagens, idades_reais in loop_treino:
            imagens = imagens.to(device)
            idades_reais = idades_reais.to(device)

            otimizador.zero_grad()
            idades_previstas = modelo(imagens)
            erro = criterio(idades_previstas, idades_reais)

            erro.backward()
            otimizador.step()

            erro_treino_total += erro.item()
            loop_treino.set_description(f"Treino Época [{epoca+1}/{num_epocas}]")

        erro_medio_treino = erro_treino_total / len(dataloader_treino)

        # ---------------- FASE DE VALIDAÇÃO ----------------
        modelo.eval() #desativa dropout e fixa batchNorm
        erro_val_total = 0.0

        #não calcular gradientes na validação
        with torch.no_grad():
            for imagens_val, idades_reais_val in dataloader_val:
                imagens_val = imagens_val.to(device)
                idades_reais_val = idades_reais_val.to(device)

                idades_previstas_val = modelo(imagens_val)
                erro_val = criterio(idades_previstas_val, idades_reais_val)

                erro_val_total += erro_val.item()

        erro_medio_val = erro_val_total / len(dataloader_val)

        print(f"Época {epoca+1}/{num_epocas} - Erro Treino: {erro_medio_treino:.2f} | Erro Validação: {erro_medio_val:.2f}")

        #salva o modelo caso o erro de validação seja o menor até agora
        if erro_medio_val < melhor_erro_val:
            melhor_erro_val = erro_medio_val
            torch.save(modelo.state_dict(), '/content/drive/MyDrive/melhor_modelo_openbhb.pth')
            print(f" --> Novo melhor modelo salvo! (Melhoria na validação para {melhor_erro_val:.2f})")

    print("\nTreinamento e Validação Concluídos!")
    print(f"Melhor erro médio de validação alcançado: {melhor_erro_val:.2f} anos.")

if __name__ == "__main__":
    treinar_modelo()

Mounted at /content/drive
Treinando na unidade: cuda
Filtrando pacientes com o nome real dos arquivos...
Filtro concluído! Encontradas 754 imagens na pasta de um total de 757.
Imagens para Treino: 603 | Imagens para Validação: 151
Iniciando o Treinamento...


Época 1/25 - Erro Treino: 23.16 | Erro Validação: 23.16
 --> Novo melhor modelo salvo! (Melhoria na validação para 23.16)


Época 2/25 - Erro Treino: 22.61 | Erro Validação: 22.85
 --> Novo melhor modelo salvo! (Melhoria na validação para 22.85)


Época 3/25 - Erro Treino: 22.17 | Erro Validação: 22.27
 --> Novo melhor modelo salvo! (Melhoria na validação para 22.27)


Época 4/25 - Erro Treino: 21.77 | Erro Validação: 21.95
 --> Novo melhor modelo salvo! (Melhoria na validação para 21.95)


...

Época 21/25 - Erro Treino: 9.34 | Erro Validação: 7.99
 --> Novo melhor modelo salvo! (Melhoria na validação para 7.99)


Época 22/25 - Erro Treino: 8.77 | Erro Validação: 8.36


Época 23/25 - Erro Treino: 7.91 | Erro Validação: 9.01


Época 24/25 - Erro Treino: 7.59 | Erro Validação: 7.85
 --> Novo melhor modelo salvo! (Melhoria na validação para 7.85)


Época 25/25 - Erro Treino: 7.09 | Erro Validação: 9.93

Treinamento e Validação Concluídos!
Melhor erro médio de validação alcançado: 7.85 anos.
